# 07 - 语雀 (Yuque) 内部 Web API 测试

> 基于 Cookie 认证的语雀内部 Web API，完全免费，无需超级会员。

## 与 Open API v2 的区别

| 对比项 | Open API v2 | 内部 Web API |
|--------|------------|-------------|
| 认证方式 | `X-Auth-Token` | `Cookie: _yuque_session=xxx; _ctoken=xxx` |
| 需要会员 | ✅ 超级会员 | ❌ 完全免费 |
| 端点前缀 | `/api/v2/...` | `/api/...` |
| 写操作 CSRF | 不需要 | 需要 `X-CSRF-Token` + `Referer` |

## 获取 Cookie 方法

1. 登录语雀网页版（https://www.yuque.com）
2. 按 F12 打开开发者工具
3. 切换到 **Application**（应用）标签
4. 左侧点击 **Cookies** → `https://www.yuque.com`
5. 复制以下两个值：
   - `_yuque_session` 的 Value
   - `_ctoken` 的 Value
6. 粘贴到 `.env` 文件：
   ```
   YUQUE_SESSION=复制的_yuque_session值
   YUQUE_CTOKEN=复制的_ctoken值
   ```

In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# 加载 .env（如果编码问题，直接手动填写）
try:
    load_dotenv(Path('../../.env'))
except:
    pass

SESSION = os.getenv('YUQUE_SESSION') or '你的_yuque_session'
CTOKEN = os.getenv('YUQUE_CTOKEN') or '你的_ctoken'

BASE = 'https://www.yuque.com'
COOKIE = f'_yuque_session={SESSION}; _ctoken={CTOKEN}'

def build_headers(referer=None):
    h = {
        'Cookie': COOKIE,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json, text/plain, */*',
        'X-CSRF-Token': CTOKEN,
        'X-Requested-With': 'XMLHttpRequest',
    }
    if referer:
        h['Referer'] = referer
    return h

def api(method, path, data=None, query=None, referer=None):
    url = f'{BASE}{path}'
    if query:
        url += '?' + '&'.join(f'{k}={v}' for k,v in query.items())
    h = build_headers(referer)
    if data and method not in ('GET', 'DELETE'):
        h['Content-Type'] = 'application/json'
    if method == 'GET':
        r = requests.get(url, headers=h)
    elif method == 'POST':
        r = requests.post(url, headers=h, json=data)
    elif method == 'PUT':
        r = requests.put(url, headers=h, json=data)
    elif method == 'DELETE':
        r = requests.delete(url, headers=h)
    return r.json()

print('Ready!')

## 1. 列出知识库 (GET /api/books)

In [ ]:
r = api('GET', '/api/books')
books = r.get('data', [])
print(f'知识库数量: {len(books)}')
for b in books[:5]:
    print(f"  ID:{b['id']} Slug:{b['slug']} Name:{b['name']}")

# 保存第一个知识库 ID 用于后续测试
if books:
    BOOK_ID = books[0]['id']
    BOOK_SLUG = books[0]['slug']
    print(f'\n使用知识库: ID={BOOK_ID}, Slug={BOOK_SLUG}')

## 2. 获取目录 (GET /api/books/{id}/toc)

In [ ]:
r = api('GET', f'/api/books/{BOOK_ID}/toc')
toc = r.get('data', {}).get('toc', [])
print(f'目录项数量: {len(toc)}')
for item in toc[:10]:
    indent = '  ' * (item.get('depth', 0))
    print(f"{indent}[{item['type']}] {item['title']}")

# 找一个 DOC 类型的条目用于读取
doc_items = [i for i in toc if i['type'] == 'DOC']
if doc_items:
    DOC_SLUG = doc_items[0]['url']
    print(f'\n使用文档 Slug: {DOC_SLUG}')

## 3. 读取文档 (GET /api/docs/{slug}?book_id={id})

In [ ]:
r = api('GET', f'/api/docs/{DOC_SLUG}', query={'book_id': str(BOOK_ID)})
doc = r.get('data', {})
print(f"标题: {doc.get('title')}")
print(f"ID: {doc.get('id')}")
print(f"格式: {doc.get('format')}")
content = doc.get('content', '')
print(f"内容长度: {len(content)} 字符")
print(f"\n内容预览:\n{content[:500]}")

## 4. 创建文档 (POST /api/docs)

In [ ]:
from datetime import datetime

r = api('POST', '/api/docs', data={
    'book_id': BOOK_ID,
    'title': f'API测试文档-{datetime.now().strftime("%H%M%S")}',
    'body': '<!doctype lake><h1>测试标题</h1><p>这是一段测试内容。</p>',
    'format': 'lake',
    'public': 0,
}, referer=f'{BASE}/{BOOK_ID}')

if 'data' in r:
    created = r['data']
    TEST_DOC_ID = created['id']
    TEST_DOC_SLUG = created['slug']
    print(f"创建成功! ID={TEST_DOC_ID}, Slug={TEST_DOC_SLUG}")
    print(f"标题: {created['title']}")
else:
    print(f"创建失败: {r}")

## 5. 更新文档 (PUT /api/docs/{id})

In [ ]:
r = api('PUT', f'/api/docs/{TEST_DOC_ID}', data={
    'title': 'API测试文档 [已更新]',
    'body': '<!doctype lake><h1>更新后的标题</h1><p>这是更新后的内容。</p><p>新增段落。</p>',
    'format': 'lake',
}, referer=f'{BASE}/{BOOK_ID}')

if 'data' in r:
    updated = r['data']
    print(f"更新成功! 标题: {updated['title']}")
else:
    print(f"更新失败: {r}")

## 6. 删除文档 (DELETE /api/docs/{id}?book_id={id})

In [ ]:
r = api('DELETE', f'/api/docs/{TEST_DOC_ID}', query={'book_id': str(BOOK_ID)}, referer=f'{BASE}/{BOOK_ID}')

if 'data' in r:
    print(f"删除成功! 文档 ID: {TEST_DOC_ID}")
else:
    print(f"删除失败: {r}")

## 附录: 端点总结

### 读取操作（无需 CSRF）
| 方法 | 端点 | 说明 |
|------|------|------|
| GET | `/api/books` | 列出知识库 |
| GET | `/api/books/{id}/toc` | 获取目录 |
| GET | `/api/docs/{slug}?book_id={id}` | 读取文档 |

### 写操作（需要 CSRF: X-CSRF-Token + Referer）
| 方法 | 端点 | Body |
|------|------|------|
| POST | `/api/docs` | `{book_id, title, body, format, public}` |
| PUT | `/api/docs/{id}` | `{title, body, format}` |
| DELETE | `/api/docs/{id}?book_id={id}` | - |

### 请求头模板
```python
headers = {
    'Cookie': '_yuque_session=xxx; _ctoken=xxx',
    'User-Agent': 'Mozilla/5.0 ...',
    'Accept': 'application/json, text/plain, */*',
    'X-CSRF-Token': 'xxx',  # _ctoken 的值
    'X-Requested-With': 'XMLHttpRequest',
    'Referer': 'https://www.yuque.com/{book_id}',  # 写操作必需
}
```